# Step 1: Import all necessary libraries

In [ ]:
# Libraries
import pandas as pd
from traffic.data import aircraft
import numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)


## Step 2: Import existing bottleneck features

In [ ]:
helper_bottleneck_features = pd.read_parquet('data/processed/helper_bottleneck_features.parquet')

In [ ]:
helper_bottleneck_features.head()

## Step 3: Import processed trajectories

In [ ]:
trajs_processed = pd.read_parquet('data/processed/trajs_processed.parquet')

In [ ]:
trajs_processed.head()

# Step 4: Import processed meteorological data

In [ ]:
meteo_processed = pd.read_parquet('data/processed/meteo_processed.parquet')

## Step 5: Feature Engineering

In the following steps, additional features based on the processed trajectories are being created and then merged with the existing bottleneck features.

### Step 5.1: Operational Features

First, we create a features dataframe and define the target variable (taxi out time) for each flight_id.

In [ ]:
features_df = helper_bottleneck_features[
    ["flight_id", "taxi_start", "taxi_end"]
].copy()

features_df["taxi_time_s"] = (
    features_df["taxi_end"] - features_df["taxi_start"]
).dt.total_seconds()

Before defining a filtering threshold, the distribution of taxi-out times is inspected using summary statistics and lower quantiles to understand the realistic range of values.

In [ ]:
def inspect_taxi_time_distribution(features_df):

    print("Summary statistics:")
    print(features_df["taxi_time_s"].describe())

    print("\nLower quantiles:")
    print(
        features_df["taxi_time_s"].quantile(
            [0.01, 0.02, 0.05, 0.10]
        )
    )


inspect_taxi_time_distribution(features_df)

A configurable minimum taxi-out time parameter is defined to allow flexible adjustment of the filtering threshold as the dataset grows or data characteristics change.


 <font color='red'> CHANGE THE PARAMETER IN THE CELL BELOW: </font>

In [ ]:
MIN_TAXI_TIME_S = 60

Flights with taxi-out times below the defined minimum threshold are excluded from further analysis to remove incomplete or operationally implausible trajectories.

In [ ]:
features_df = features_df[
    features_df["taxi_time_s"] >= MIN_TAXI_TIME_S
]

Adding aircraft typecode as a feature and setting datatype as "category"

In [ ]:
typecode_lookup = (
    trajs_processed[["flight_id", "typecode"]]
    .dropna(subset=["typecode"])
    .drop_duplicates()
)

features_df = features_df.merge(
    typecode_lookup,
    on="flight_id",
    how="left"
)

features_df["typecode"] = features_df["typecode"].astype("category")

The queue size feature is calculated within a rolling one-hour lookback window. For each flight, only aircraft that started taxiing within the previous hour and were still taxiing at the current flight's taxi_start time are counted. This keeps the feature operationally realistic, prediction-safe, and computationally scalable for larger datasets.

In [ ]:
WINDOW = pd.Timedelta(hours=1)

features_df = features_df.sort_values("taxi_start")

starts = features_df["taxi_start"]
ends = features_df["taxi_end"]

queue_size_last_hour = []

for i, t in enumerate(starts):
    n_active = (
        (starts >= t - WINDOW)
        & (starts < t)
        & (ends > t)
    ).sum()

    queue_size_last_hour.append(n_active)

features_df["queue_size"] = queue_size_last_hour

Implementing the average taxi out time of the past 6 hours as a feature

In [ ]:
WINDOW = pd.Timedelta(hours=6)

features_df = features_df.sort_values("taxi_start")

starts = features_df["taxi_start"]
ends = features_df["taxi_end"]
taxi_times = features_df["taxi_time_s"]

avg_taxi_out_6h = []

for i, t in enumerate(starts):

    mask = (
        (ends >= t - WINDOW)
        & (ends < t)
    )

    if mask.any():
        avg = taxi_times[mask].mean()
    else:
        avg = None

    avg_taxi_out_6h.append(avg)

features_df["avg_taxi_out_time_6h"] = avg_taxi_out_6h

Implementing the average taxi out time of the past hour as a feature.

In [ ]:
WINDOW = pd.Timedelta(hours=1)

features_df = features_df.sort_values("taxi_start")

starts = features_df["taxi_start"]
ends = features_df["taxi_end"]
taxi_times = features_df["taxi_time_s"]

avg_taxi_out_1h = []

for i, t in enumerate(starts):

    mask = (
        (ends >= t - WINDOW)
        & (ends < t)
    )

    if mask.any():
        avg = taxi_times[mask].mean()
    else:
        avg = None

    avg_taxi_out_1h.append(avg)

features_df["avg_taxi_out_time_1h"] = avg_taxi_out_1h

Implementing the average taxi out time of the past 15 minutes as feature.

In [ ]:
WINDOW = pd.Timedelta(hours=0.25)

features_df = features_df.sort_values("taxi_start")

starts = features_df["taxi_start"]
ends = features_df["taxi_end"]
taxi_times = features_df["taxi_time_s"]

avg_taxi_out_15min = []

for i, t in enumerate(starts):

    mask = (
        (ends >= t - WINDOW)
        & (ends < t)
    )

    if mask.any():
        avg = taxi_times[mask].mean()
    else:
        avg = None

    avg_taxi_out_15min.append(avg)

features_df["avg_taxi_out_time_15min"] = avg_taxi_out_15min

Implementing bottleneck features

In [ ]:
cols_to_merge = [
    "flight_id",
    "n_intersections_passed",
    "sum_seg_length_m",
    "sum_p10_hist_seg_time",
    "sum_median_hist_seg_time",
    "sum_p90_hist_seg_time",
    "TO_runway_28",
    "TO_runway_32",
    "TO_runway_16",
    "TO_runway_10",
    "TO_runway_34",
    "pushback",
]

features_df = features_df.merge(
    helper_bottleneck_features[cols_to_merge],
    on="flight_id",
    how="left",
)

### Step 5.2 Temporal Features

Defining a function for cyclic encoding

In [ ]:
def sin_cos(x,T):
    return np.sin(2 * np.pi * x / T), np.cos(2 * np.pi * x / T)

Splitting up information from the 'taxi_start' column into month, day of week and minute of day

In [ ]:
features_df["month"] = features_df["taxi_start"].dt.month

features_df["day_of_week"] = features_df["taxi_start"].dt.dayofweek
# Monday = 0, Sunday = 6

features_df["minute_of_day"] = (
    features_df["taxi_start"].dt.hour * 60
    + features_df["taxi_start"].dt.minute
)

Encoding month, day of week and minute of day as sin and cos

In [ ]:
features_df["month_sin"], features_df["month_cos"] = sin_cos(
    features_df["month"], 12
)

features_df["day_of_week_sin"], features_df["day_of_week_cos"] = sin_cos(
    features_df["day_of_week"], 7
)

features_df["minute_of_day_sin"], features_df["minute_of_day_cos"] = sin_cos(
    features_df["minute_of_day"], 24 * 60
)

### Step 5.3: Meteorological Features

Implementing temperature and spread as  features

In [ ]:
meteo_temp = (
    meteo_processed[
        ["reference_timestamp", "tre200s0", "spread"]
    ]
    .rename(
        columns={
            "tre200s0": "temp_at_taxi_start",
            "spread": "spread_at_taxi_start"
        }
    )
)

features_df = pd.merge_asof(
    features_df.sort_values("taxi_start"),
    meteo_temp.sort_values("reference_timestamp"),
    left_on="taxi_start",
    right_on="reference_timestamp",
    direction="backward"
)

features_df = features_df.drop(columns="reference_timestamp")

In [ ]:
features_df.head(6)

Export features as parquet file

In [ ]:
output_path = "data/processed/features.parquet"
 
# Save dataframe to pkl.
features_df.to_parquet(output_path)
 
print(f"File \"features\" saved to: {output_path}")
 